# U6-03 · Densidad, mezclas y reducción de dimensionalidad

*Machine Learning e Inteligencia Artificial (IFCD0174) · Unidad 6, apartados 6.5 a 6.7*

Ejecuta cada celda con **Mayús + Intro**. Los conjuntos de datos vienen incluidos en scikit-learn; no hace falta descargar nada.

## 1. Formas que k-medias no ve (ejemplo 6 y figura 6.4)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import adjusted_rand_score

Xm, ym = make_moons(n_samples=400, noise=0.07, random_state=0)
km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(Xm)
db = DBSCAN(eps=0.2, min_samples=5).fit(Xm)
gm = GaussianMixture(n_components=2, random_state=0).fit(Xm)
print("k-medias ARI:", round(adjusted_rand_score(ym, km.labels_), 3))
print("DBSCAN   ARI:", round(adjusted_rand_score(ym, db.labels_), 3), "· puntos de ruido:", (db.labels_ == -1).sum())
print("mezcla   ARI:", round(adjusted_rand_score(ym, gm.predict(Xm)), 3))
fig, axs = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, (t, l) in zip(axs, {"k-medias": km.labels_, "DBSCAN": db.labels_, "mezcla gaussiana": gm.predict(Xm)}.items()):
    ax.scatter(Xm[:, 0], Xm[:, 1], c=l, s=8, cmap="coolwarm"); ax.set_title(t)
plt.tight_layout(); plt.show()

**Prueba.** Cambia `eps` a 0.1 y a 0.4. ¿Qué le pasa al número de grupos y de puntos de ruido?

## 2. PCA de los vinos (ejemplo 7 y figura 6.5)

In [ ]:
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X, y = load_wine(return_X_y=True, as_frame=True)
Z = StandardScaler().fit_transform(X)
pca = PCA().fit(Z)
print("varianza explicada:", pca.explained_variance_ratio_[:3].round(3))
print("acumulada         :", pca.explained_variance_ratio_.cumsum()[:8].round(3))
cargas = pd.DataFrame(pca.components_[:2].T, index=X.columns, columns=["CP1", "CP2"])
cargas.round(2).sort_values("CP1")

In [ ]:
Z2 = pca.transform(Z)[:, :2]
km2 = KMeans(n_clusters=3, n_init=10, random_state=0).fit(Z2)
print("k-medias sobre 2 componentes: ARI =", round(adjusted_rand_score(y, km2.labels_), 3))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.8))
a1.bar(range(1, 14), pca.explained_variance_ratio_)
a1.plot(range(1, 14), pca.explained_variance_ratio_.cumsum(), marker="o", color="k")
a1.axhline(0.8, ls=":", color="gray"); a1.set_xlabel("componente")
for c in range(3):
    a2.scatter(*Z2[y == c].T, s=12, label=f"cultivar {c + 1}")
a2.set_xlabel("CP1"); a2.set_ylabel("CP2"); a2.legend()
plt.tight_layout(); plt.show()

## 3. t-SNE (apartado 6.7)

t-SNE solo sirve para **mirar**: las distancias entre grupos y sus tamaños en el mapa no significan nada. UMAP no está disponible en este entorno del navegador; en tu ordenador se instala con `pip install umap-learn`.

In [ ]:
from sklearn.manifold import TSNE
fig, axs = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, perp in zip(axs, [5, 30, 50]):
    T = TSNE(n_components=2, perplexity=perp, random_state=0).fit_transform(Z)
    ax.scatter(T[:, 0], T[:, 1], c=y, s=10, cmap="viridis"); ax.set_title(f"perplejidad = {perp}")
plt.tight_layout(); plt.show()

**Ejercicio.** Cambia `random_state` en t-SNE y vuelve a ejecutar. ¿Qué cambia en el mapa y qué se mantiene?

> Schubert, E., Sander, J., Ester, M., Kriegel, H.-P. y Xu, X. (2017). DBSCAN Revisited, Revisited: Why and How You Should (Still) Use DBSCAN. *ACM Transactions on Database Systems*, 42(3), 19. https://doi.org/10.1145/3068335
>
> Jolliffe, I. T. y Cadima, J. (2016). Principal component analysis: a review and recent developments. *Phil. Trans. R. Soc. A*, 374, 20150202. https://doi.org/10.1098/rsta.2015.0202
>
> Wattenberg, M., Viégas, F. y Johnson, I. (2016). How to Use t-SNE Effectively. *Distill*. https://doi.org/10.23915/distill.00002